# Democratizing Data for Ghost Kitchen Operations with AI/BI Genie

Ghost kitchen operations generate rich operational data, but accessing insights traditionally requires technical expertise. Business users need answers to questions like "What's our average delivery time?" or "Which menu items are underperforming?" but often can't get them without SQL knowledge or analyst support.

**The Challenge:**
- Business questions require technical SQL knowledge or waiting for analyst support
- Complex database schemas don't match business terminology
- Critical insights are locked behind technical barriers

**The Solution:** Databricks AI/BI Genie Spaces enable any business user to ask questions in natural language and receive immediate insights and visualizations. More importantly, Genie can learn your specific business logic and definitions, making it even more valuable for domain-specific analysis.

## Key Capabilities

- **Natural Language Queries:** Ask questions in plain English like "Show me late deliveries by location" instead of writing complex SQL.
- **Business Logic Integration:** Teach Genie your specific definitions (like how you define "late" orders) so it applies your business rules consistently.
- **Self-Service Analytics:** Business users get instant insights without depending on analysts or engineers.

This demo shows how to set up AI/BI Genie for ghost kitchen data and configure it with business logic to maximize its value for operational decision-making.

## Demo Overview

This demo demonstrates how to make ghost kitchen operational data accessible to business users through natural language queries. We'll walk through:

1. **Data Foundation Setup**: Initialize the catalog and load ghost kitchen delivery event data
2. **Genie Space Creation**: Connect AI/BI Genie to your operational data
3. **Natural Language Queries**: Ask business questions without writing SQL
4. **Business Logic Training**: Teach Genie your specific definitions and business rules

The demo builds on delivery event data that captures the complete order lifecycle from creation through final delivery.

## Setup

This demo builds on the data foundation created in the [Lakeflow demo](lakeflow.ipynb). If you've already completed the Lakeflow demo, the setup below will detect your existing data and skip redundant steps.

If you haven't run the Lakeflow demo yet, this setup will:
- Create a Unity Catalog named `gk_demo` with a `default` schema
- Set up a volume for raw data storage
- Load sample Ghost Kitchen delivery event data into a Delta table

The setup ensures you have the `gk_demo.default.all_events` table needed for this AI/BI demonstration.

In [0]:
from utils.utils import (
    setup_catalog_and_volume,
    copy_raw_data_to_volume,
    initialize_events_table,
    drop_gk_demo_catalog,
)

# Drop existing catalog/volume/table if you need to start fresh
# drop_gk_demo_catalog(spark)

## Setup the catalog and volume
setup_catalog_and_volume(spark)

## Copy the raw data to the volume
copy_raw_data_to_volume()

## Initialize the events table
initialize_events_table(spark)


## Create a New Genie Space

The above cell generated a table of raw delivery events called `gk_demo.default.all_events`. Now, let's create a _Genie Space_ over the data. To create a Genie space, click "Genie" in the left sidebar, then click the "+ New" button.

<img src="./images/ai_bi_genie/1_new_genie.png" width="75%">

You will be prompted to connect your data to the Genie space. If you have recently accessed the `all_events` table, it might show up in the list of recent tables in the "For you" section. Otherwise, click "All" and navigate to find the table by clicking through the Unity Catalog hierarchy:
1. First click on **gk_demo** (the catalog)
2. Then click on **default** (the schema/database)  
3. Finally select **all_events** (the table)

The full table name is `gk_demo.default.all_events` where the dots represent the hierarchy levels: `catalog.schema.table`. Once you've selected the table, click "Create."

<img src="./images/ai_bi_genie/2_select_data.png" width="75%">

## Ask questions about the data

Now you can immediately start asking questions about the dataset in natural language! Try it out. You might ask:
- What is the date range covered by the data?
- What are the different event statuses each order can have?
- Create a histogram of delivery times.

Genie will generate a SQL query to attempt to answer these questions. You can see the query by clicking "Show Code."

**Note:** AI/BI Genie responses can vary between requests due to the non-deterministic nature of large language models. If you don't get the expected result on your first try, you can rephrase your question or provide more specific instructions. The following sections demonstrate one possible path through the demo, but your experience may vary.

<img src="./images/ai_bi_genie/3_ask_genie.png" width="75%">


## Improve Genie's Performance

Let's try out a harder version of the last sample query above: "Create a histogram of the delivery times for each order, with the bins colored based on whether the delivery was on time or late. Bin by the minute."

There are a number of things that might go wrong here—again, Genie's responses are not deterministic, so you might find that it takes some additional prompting to get a working histogram, or that its interpretation of "late" is different from yours. In the next sections, we will show how to steer Genie's responses by providing business logic and sample queries.

<img src="./images/ai_bi_genie/4_late_attempt_1.png" width="75%">


### Improve Genie's performance by providing business logic and definitions

In our case, Genie picked an arbitrary cutoff for "late" at 30 minutes, classifying a significant proportion of deliveries as late! Perhaps we internally think of late orders as those exceeding the P95 delivery time. This is not explicitly encoded anywhere in the data. But we can tell Genie that this is the business logic we use to determine whether an order is late, and it will use this information to make the correct graph. To add custom instructions, select "Configure," navigate to "Instructions," and add any relevant business logic. Let's try it out.

<img src="./images/ai_bi_genie/5_custom_instructions.png" width="75%">

```
Definitions:
- *Late* orders are those that took longer than the P95 delivery time.
- *Delivery Time* refers to the total order time, from order creation through delivery.
- When asked about order timing, round to the
nearest minute.
```

Paste the custom instructions above and click "Save" to save them. Now Genie will know the definition of a "Late" order and be able to use that definition to answer future questions. We added a couple of additional instructions to help steer the responses as well:
- Genie should interpret "delivery time" as the total order time, from creation to delivery.
- Genie should round all timings to the nearest minute.

### Save example queries

Now let's ask Genie to identify the cutoff for a late order: **"What is the p95 delivery time?"**

Genie will write and execute a SQL query to compute the 95th percentile delivery time (around 36 minutes). You can see the generated SQL by clicking "Show Code" in the chat.

Once you see the results, you can click **"Add as instruction"** to save both the question and the specific SQL query that Genie generated. This teaches Genie how to calculate the p95 delivery time for future questions, improving its performance when answering related queries about late orders.

<img src="./images/ai_bi_genie/6_save_query.png" width="75%">

Equipped with this context—and a sample query showing how to operationalize it—Genie can now generate the histogram we asked for.

<img src="./images/ai_bi_genie/7_late_attempt_2.png" width="75%">

Notice the transformation: we went from Genie making arbitrary assumptions (30-minute cutoff) to using precise business logic (P95 definition). By adding custom instructions and saving example queries, we've taught Genie the specific domain knowledge it needs to provide accurate, business-relevant insights about Ghost Kitchen operations.


## Next Steps

We have seen how to use AI/BI Genie to answer natural language questions about data, and how to improve Genie's performance via custom instructions and sample SQL queries. There are many more ways to use and improve Genie. For example, you can:
- Use [benchmarks](https://docs.databricks.com/aws/en/genie/benchmarks) to create sets of test questions for assessing response accuracy
- [Review Responses](https://docs.databricks.com/aws/en/genie/set-up#review-responses) in the "Monitoring" interface
- [Edit Metadata](https://docs.databricks.com/aws/en/genie/set-up#edit-knowledge-store-metadata) to update Genie's knowledge about your stored data

You can learn more about best practices for Genie spaces [here](https://docs.databricks.com/aws/en/genie/best-practices).